# MP4 から before / after 候補を選ぶ

この notebook は `analysis/run_video_roi_search.py` を操作するための薄い入口です。
肌色・Lab値では候補を順位付けせず、顔向き・顔サイズ・ROI形状・目の開き・口の開きなどの幾何条件で before / after 候補を探します。

候補抽出 → 目視確認 → `selected_pair.json` で固定、という順に進みます。Lab解析は固定済みペアに対してだけ、後半の明示的なセルで実行します。


In [ ]:
from pathlib import Path
import os

cwd = Path.cwd().resolve()
marker = Path('analysis/run_video_roi_search.py')

if (cwd / marker).is_file():
    REPO_ROOT = cwd
elif cwd.name == 'notebooks' and (cwd.parent / marker).is_file():
    REPO_ROOT = cwd.parent
    os.chdir(REPO_ROOT)
else:
    raise RuntimeError(
        'ikiikimake のリポジトリ直下、またはその notebooks/ から実行してください。'
        f' current={cwd}'
    )

print('repo root:', Path.cwd())

In [ ]:

# ---- 実験条件 ----
VIDEO = Path('makeup2.mp4')
OUTPUT = Path('outputs/makeup_video_search_notebook')

# 秒。動画内で「メイク前」「完成後」と確認できた区間だけを指定する。
BEFORE_RANGE = (65.0, 196.0)
AFTER_RANGE = (2022.0, 2124.0)

INTERVAL = 5.0
REFINE_INTERVAL = 1.0
TOP = 10

# 目視確認前は None のまま。候補1を採用するなら後で 1 に変更する。
APPROVED_RANK = None


In [ ]:
import math
import shutil

# fail-fast: 実行場所・入力・外部コマンドを先に検証する。
if not Path('analysis/run_video_roi_search.py').is_file():
    raise RuntimeError('analysis/run_video_roi_search.py が見つかりません。リポジトリ構成を確認してください。')
if not VIDEO.is_file():
    raise FileNotFoundError(f'動画が見つかりません: {VIDEO}')
if shutil.which('ffprobe') is None:
    raise RuntimeError('ffprobe が PATH にありません。動画長を安全に取得できないため停止します。')
if not all(math.isfinite(v) for v in (*BEFORE_RANGE, *AFTER_RANGE, INTERVAL, REFINE_INTERVAL)):
    raise ValueError('時刻・間隔は有限値で指定してください。')
if not (0 <= BEFORE_RANGE[0] < BEFORE_RANGE[1] < AFTER_RANGE[0] < AFTER_RANGE[1]):
    raise ValueError('BEFORE_RANGE と AFTER_RANGE は重ならない昇順区間で指定してください。')
if INTERVAL <= 0 or REFINE_INTERVAL <= 0 or REFINE_INTERVAL > INTERVAL:
    raise ValueError('0 < REFINE_INTERVAL <= INTERVAL を満たしてください。')
if TOP < 1:
    raise ValueError('TOP は1以上にしてください。')

print('preflight OK')

## 候補探索を実行

`run_video_roi_search.py` と同じ処理を呼びます。粗探索は `BEFORE_RANGE` と `AFTER_RANGE` の中だけを走査し、動画の途中の未使用区間は解析しません。
既存の非空出力フォルダには上書きしません。
途中のROI生成に失敗したフレームは理由を記録して候補から除外しますが、入力形式や処理方法を別方式へ切り替えるフォールバックはしません。


In [ ]:
from analysis.run_video_roi_search import parse_args, run

argv = [
    '--video', str(VIDEO),
    '--output', str(OUTPUT),
    '--before-range', str(BEFORE_RANGE[0]), str(BEFORE_RANGE[1]),
    '--after-range', str(AFTER_RANGE[0]), str(AFTER_RANGE[1]),
    '--interval', str(INTERVAL),
    '--refine-interval', str(REFINE_INTERVAL),
    '--top', str(TOP),
]

manifest = run(parse_args(argv))
if manifest['status'] != 'needs_review':
    raise RuntimeError(f"比較候補を作れませんでした: status={manifest['status']}")

print('候補生成完了:', OUTPUT / 'report.html')

## 最良候補を表示

ここで必ず目視します。頬・額ROI、髪・手・影・道具、顔向き、表情、before/afterの意味が妥当かを確認してください。


In [ ]:
from IPython.display import display, Image

for name in ('best_pair_faces.png', 'best_pair_roi_overlay.png'):
    path = OUTPUT / name
    if not path.is_file():
        raise FileNotFoundError(f'レビュー画像がありません: {path}')
    display(Image(filename=str(path)))

## 上位候補の数値を確認

スコアは「幾何的な差」で、小さいほど条件が近い候補です。美しさ・メイク効果・本人一致の確率ではありません。


In [ ]:
import json
from IPython.display import HTML, display

matching_path = OUTPUT / 'matching.json'
if not matching_path.is_file():
    raise FileNotFoundError(matching_path)

matching = json.loads(matching_path.read_text(encoding='utf-8'))
pairs = matching.get('ranked_pairs', [])
if not pairs:
    raise RuntimeError('ranked_pairs が空です。')

columns = [
    ('rank', 'rank'), ('before_time', 'before_time'), ('after_time', 'after_time'),
    ('score', 'score'), ('yaw_gap', 'yaw_gap_degrees'), ('pitch_gap', 'pitch_gap_degrees'),
    ('roll_gap', 'roll_gap_degrees'), ('face_scale_ratio', 'face_scale_ratio'),
    ('roi_rms', 'roi_procrustes_rms'), ('eye_gap', 'eye_aperture_gap'), ('mouth_gap', 'mouth_opening_gap'),
]

headers = ''.join(f'<th>{label}</th>' for label, _ in columns)
body = []
for rank, pair in enumerate(pairs, 1):
    terms = pair['terms']
    values = {
        'rank': rank,
        'before_time': pair['before_time'],
        'after_time': pair['after_time'],
        'score': pair['score'],
        **terms,
    }
    cells = ''.join(f'<td>{values[key]:.4f}</td>' if isinstance(values[key], float) else f'<td>{values[key]}</td>' for _, key in columns)
    body.append(f'<tr>{cells}</tr>')

display(HTML(f'<table><thead><tr>{headers}</tr></thead><tbody>{"".join(body)}</tbody></table>'))

## 上位候補を画像で全部確認

`best_pair_*` は1位だけです。ここでは `TOP` 件の before / after 元画像と ROI overlay を順番に表示します。
ブレ・手・髪・表情・テロップ・ROI位置を見て、1位以外に使える組がないか確認してください。


In [ ]:
from IPython.display import display, HTML, Image

scan_manifest = json.loads((OUTPUT / 'scan_manifest.json').read_text(encoding='utf-8'))
records_by_id = {record['frame_id']: record for record in scan_manifest['records']}

for rank, pair in enumerate(pairs, 1):
    before = records_by_id[pair['before_id']]
    after = records_by_id[pair['after_id']]
    display(HTML(
        f"<h3>候補 {rank} / score={pair['score']:.4f} / "
        f"before={pair['before_time']:.2f}s / after={pair['after_time']:.2f}s</h3>"
    ))

    for phase, record in (('BEFORE', before), ('AFTER', after)):
        image_path = Path(record['image_path'])
        overlay_path = Path(record['roi_dir']) / 'roi_overlay.png'
        if not image_path.is_file():
            raise FileNotFoundError(image_path)
        if not overlay_path.is_file():
            raise FileNotFoundError(overlay_path)
        display(HTML(f'<b>{phase} 元画像</b>'))
        display(Image(filename=str(image_path), width=520))
        display(HTML(f'<b>{phase} ROI</b>'))
        display(Image(filename=str(overlay_path), width=520))


## 目視承認したペアを固定

上の設定セルで `APPROVED_RANK = 1` のように変更してから実行します。`None` のままなら停止します。
このセルはLab解析をしません。採用したフレームID・時刻・元画像・ROIディレクトリ・SHA-256を `selected_pair.json` に固定します。


In [ ]:
import hashlib

if APPROVED_RANK is None:
    raise RuntimeError('目視確認後に APPROVED_RANK を設定してください。自動承認はしません。')
if isinstance(APPROVED_RANK, bool) or not isinstance(APPROVED_RANK, int):
    raise TypeError('APPROVED_RANK は整数で指定してください。')
if not (1 <= APPROVED_RANK <= len(pairs)):
    raise ValueError(f'APPROVED_RANK は 1..{len(pairs)} の範囲です。')

pair = pairs[APPROVED_RANK - 1]
scan_manifest = json.loads((OUTPUT / 'scan_manifest.json').read_text(encoding='utf-8'))
records = {record['frame_id']: record for record in scan_manifest['records']}

def sha256(path: Path) -> str:
    if not path.is_file():
        raise FileNotFoundError(path)
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

selected = {'rank': APPROVED_RANK, 'score': pair['score'], 'before': {}, 'after': {}}
for phase, key in (('before', 'before_id'), ('after', 'after_id')):
    record = records.get(pair[key])
    if record is None:
        raise RuntimeError(f"matching.json の {phase} フレームが scan_manifest.json にありません: {pair[key]}")
    image_path = Path(record['image_path'])
    roi_dir = Path(record['roi_dir'])
    masks_path = roi_dir / 'roi_masks.npz'
    points_path = roi_dir / 'roi_points.json'
    overlay_path = roi_dir / 'roi_overlay.png'
    for required in (image_path, masks_path, points_path, overlay_path):
        if not required.is_file():
            raise FileNotFoundError(required)
    selected[phase] = {
        'frame_id': record['frame_id'],
        'timestamp_seconds': record['timestamp_seconds'],
        'image_path': str(image_path),
        'image_sha256': sha256(image_path),
        'roi_dir': str(roi_dir),
        'roi_masks_sha256': sha256(masks_path),
        'roi_points_sha256': sha256(points_path),
        'roi_overlay_sha256': sha256(overlay_path),
    }

selected_path = OUTPUT / 'selected_pair.json'
if selected_path.exists():
    raise FileExistsError(f'既に選択結果があります。上書きしません: {selected_path}')
selected_path.write_text(json.dumps(selected, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('固定しました:', selected_path)
selected

## 固定したペアで Lab 解析

`selected_pair.json` に記録した元画像・ROI成果物の SHA-256 を再確認してから、既存の `analysis.analyze_cheek_lab.analyze_pair` を呼びます。
ハッシュ不一致、ファイル欠落、既存の解析出力がある場合は停止します。別経路へのフォールバックはしません。

額差し引き (`delta_minus_forehead`) は未検証の control 比較です。照明補正やメイク効果の確定値としては扱いません。


In [ ]:
from analysis.analyze_cheek_lab import analyze_pair

selected_path = OUTPUT / 'selected_pair.json'
if not selected_path.is_file():
    raise FileNotFoundError(f'固定済みペアがありません: {selected_path}')

selected = json.loads(selected_path.read_text(encoding='utf-8'))
LAB_OUTPUT = OUTPUT / 'lab_selected_pair'
if LAB_OUTPUT.exists():
    raise FileExistsError(f'解析出力が既に存在します。上書きしません: {LAB_OUTPUT}')

def verify_sha256(path: Path, expected: str, label: str) -> None:
    actual = sha256(path)
    if actual != expected:
        raise RuntimeError(
            f'{label} の SHA-256 が selected_pair.json と一致しません。'
            f' expected={expected} actual={actual} path={path}'
        )

resolved = {}
for phase in ('before', 'after'):
    item = selected.get(phase)
    if not isinstance(item, dict):
        raise ValueError(f'selected_pair.json に {phase} がありません。')

    image_path = Path(item['image_path'])
    roi_dir = Path(item['roi_dir'])
    masks_path = roi_dir / 'roi_masks.npz'
    points_path = roi_dir / 'roi_points.json'
    overlay_path = roi_dir / 'roi_overlay.png'

    verify_sha256(image_path, item['image_sha256'], f'{phase} image')
    verify_sha256(masks_path, item['roi_masks_sha256'], f'{phase} roi_masks')
    verify_sha256(points_path, item['roi_points_sha256'], f'{phase} roi_points')
    verify_sha256(overlay_path, item['roi_overlay_sha256'], f'{phase} roi_overlay')

    resolved[phase] = {'image': image_path, 'masks': masks_path}

summary = analyze_pair(
    resolved['before']['image'],
    resolved['after']['image'],
    resolved['before']['masks'],
    resolved['after']['masks'],
    LAB_OUTPUT,
)

print('Lab analysis complete:', LAB_OUTPUT)


## 頬の a* と額 control を確認

`delta` は after − before、`delta_minus_forehead` は `(頬 after − before) − (額 after − before)` です。
まず `a_median` を中心に確認し、必要なら `a_mean` や分布画像も見ます。


In [ ]:
a_rows = [
    row for row in summary['deltas']
    if row['metric'] in ('a_median', 'a_mean')
]

if not a_rows:
    raise RuntimeError('a* の差分結果がありません。')

headers = ('side', 'metric', 'before', 'after', 'delta', 'forehead_delta', 'delta_minus_forehead')
head_html = ''.join(f'<th>{name}</th>' for name in headers)
body_html = []
for row in a_rows:
    cells = []
    for name in headers:
        value = row[name]
        cells.append(f'<td>{value:.3f}</td>' if isinstance(value, float) else f'<td>{value}</td>')
    body_html.append('<tr>' + ''.join(cells) + '</tr>')

display(HTML(
    '<table><thead><tr>' + head_html + '</tr></thead><tbody>' + ''.join(body_html) + '</tbody></table>'
))

display(HTML('<p><b>ROI sample</b></p>'))
display(Image(filename=str(LAB_OUTPUT / 'roi_samples.png'), width=900))

for name in ('left_cheek', 'right_cheek', 'forehead'):
    hist = LAB_OUTPUT / f'{name}_lab_hist.png'
    if not hist.is_file():
        raise FileNotFoundError(hist)
    display(HTML(f'<p><b>{name} Lab histogram</b></p>'))
    display(Image(filename=str(hist), width=900))


## 出力

`LAB_OUTPUT` に `analysis_summary.json`, `lab_stats.csv`, `lab_deltas.csv`, `roi_samples.png`, 各ROIのLabヒストグラムが保存されます。
notebookをGitへ保存するときは、画像出力で `.ipynb` が巨大化しないよう `Clear All Outputs` してから保存します。
